# Sprint 1 Campus — From application to service

HelioDesk wants another application to call FieldCare. We will follow one question across the service boundary, then change one agreement at a time.

Use **one saved copy of this notebook throughout the sprint**. Work one demonstration at a time: predict, run, read the result, then explain the boundary it reveals. Each section names the matching Campus lesson. The optional project cells stay off until you are ready to test your own work; **Run all runs the supplied demos without requiring a finished assessment**.

| Campus lesson | Notebook destination | What to notice |
|---|---|---|
| Application vs service; REST in plain language | §1 · Application, service, and REST | A request has a method, path, body, and response. |
| Touring the provided FastAPI service | §2 · Tour the request path | Validation, clarification, retrieval, and generation are different boundaries. |
| Extending the scaffold for a new use case | §3 · Extend the scaffold | A supplied analogous route binds its own prompt; you design the dispatch route. |
| Why contracts matter | §4 · Contracts for callers | Shape validation does not establish answer quality. |
| Why and how to version endpoints | §5 · Version endpoints | The old agreement survives a stricter pilot. |
| Service scaffold checkpoint | §6 · Service scaffold checkpoint | Test your own supervisor operation. |

**Two folders, two purposes.** `DEMO` contains supplied, runnable instructor examples. `PROJECT` contains your editable service and any imported checkpoint. Demonstrations never edit `PROJECT`; a successful demo is not assessment completion. Source modules and the evaluation design are already in the repository. Cells make calls and inspect results; they do not manufacture Python files.

**Before running:** add `OPENROUTER_API_KEY` to Colab Secrets and enable notebook access, or use the hidden prompt. Initial Chroma indexing and supported model requests use provider quota. Generated wording and timings vary. Keep credentials out of cells and output.

Open Colab’s table of contents for section navigation. Save your personal copy in Drive for notes and outputs; use the final checkpoint cell to save your own service files.

## Campus lessons

- [Application vs service](https://app.notion.com/p/3ea5a55972ad81a09e8feba7b4bb576a?pvs=204)
- [REST in plain language](https://app.notion.com/p/3ea5a55972ad813b906bd2f6edb69740?pvs=204)
- [Touring the provided FastAPI service](https://app.notion.com/p/3ea5a55972ad81cd844dccbe5d8b0b89?pvs=204)
- [Extending the scaffold for a new use case](https://app.notion.com/p/3ea5a55972ad811e9461cc2e184edeb6?pvs=204)
- [Why contracts matter for AI services](https://app.notion.com/p/3ea5a55972ad812c8fcaf9462d1a2b9e?pvs=204)
- [Why and how to version AI endpoints](https://app.notion.com/p/3ea5a55972ad815795d0e07af30135e7?pvs=204)
- [Service scaffold checkpoint](https://app.notion.com/p/3ea5a55972ad8180aabdcc14a959c3d2?pvs=204)

## Set up once

Run setup in a fresh runtime. It installs the repository and opens the real FieldCare application through FastAPI’s `TestClient`, an in-process HTTP test interface. There is no network listener or `service.base_url` in this sprint’s demos. Chroma and OpenRouter still perform real retrieval/embedding and generation work.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
BASE = Path("/content") if Path("/content").exists() else Path.cwd() / "campus-runtime"
SOURCE_REVISION = "5df838530f9220af86e36224cfdfaaadccf3beb0"
REPO = BASE / f"ai-app-dev_Mod-B-{SOURCE_REVISION[:12]}"
DEMO = BASE / f"fieldcare-campus-demo-1-{SOURCE_REVISION[:12]}"
PROJECT = BASE / "fieldcare-campus-project-1"
CHECKPOINT = ""  # Optional earlier Campus source ZIP; restore to a new PROJECT path.

# Source and instructor examples are versioned; learner PROJECT files are retained.
if not REPO.exists():
    subprocess.run(["git", "init", str(REPO)], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", REPOSITORY, SOURCE_REVISION], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True, capture_output=True)
actual_revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError("Teaching source differs from the required revision. Choose a new REPO folder; existing files were preserved.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
# An older notebook may already have imported the old package in this kernel.
loaded_module = sys.modules.get("module_b")
if loaded_module is not None and Path(loaded_module.__file__).resolve().parent != (REPO / "src/module_b").resolve():
    for module_name in list(sys.modules):
        if module_name == "module_b" or module_name.startswith("module_b."):
            del sys.modules[module_name]

from module_b.openrouter import DEFAULT_CHAT_MODEL, require_openrouter_key
from module_b.workspace import prepare_example
from module_b.campus import prepare_project, save_checkpoint
require_openrouter_key(prompt=True)
os.environ.setdefault("OPENROUTER_MODEL", DEFAULT_CHAT_MODEL)
os.environ["CAMPUS_SPRINT"] = "1"
DEMO = prepare_example(destination=DEMO, repo_root=REPO)
PROJECT = prepare_project(REPO, PROJECT, CHECKPOINT)
os.chdir(DEMO)
sys.path.insert(0, str(DEMO))
if globals().get("client") is not None:
    client.__exit__(None, None, None)
    client = None
for module_name in list(sys.modules):
    if module_name == "app" or module_name.startswith("app."):
        del sys.modules[module_name]

from fastapi.testclient import TestClient
from app.campus_demo import create_app
if globals().get("client") is not None:
    client.__exit__(None, None, None)
app = create_app()
client = TestClient(app)
client.__enter__()

print("Teaching source revision:", actual_revision[:12])
print("Instructor source:", DEMO)
print("Your project:", PROJECT)
print("Project origin:", "imported checkpoint" if CHECKPOINT else "supplied starter (not a completed assessment)")

## 1. Application, service, and REST

The notebook is the caller. FieldCare is the service; FastAPI maps a method and path to a handler. **Predict:** which call needs a JSON body, and which can tell us anything about an answer?

Run both calls. Expect `200` with health metadata for `GET /health`. A supported `POST /v1/diagnose` returns an actual generated answer and citations when its provider call succeeds. Health alone cannot establish that model generation works.

In [ ]:
health = client.get("/health")
question = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
diagnosis = client.post("/v1/diagnose", json=question)
print("GET /health:", health.status_code, health.json())
print("POST /v1/diagnose:", diagnosis.status_code)
print(json.dumps(diagnosis.json(), indent=2))

**Read the result:** `200` is an HTTP outcome; `ready` or `needs_clarification` describes the application result. A cited document identifies evidence to inspect, not proof that every generated claim is supported.

In Colab’s Files panel, open `DEMO/app/routes.py`, `schemas.py`, `service.py`, `orchestration.py`, and `model.py`. These are actual supplied source files. Follow the handler → `run_diagnosis` → LangGraph → OpenRouter path. Chroma retrieves relevant documents; the model generates the answer from that context.

## 2. Tour the request path

**Predict the stop point:** an empty body is malformed; a non-empty question without equipment context is valid but cannot support model-specific guidance. We reuse the supported result from §1 rather than paying for the same demonstration twice.

Expected: missing or blank `question` → `422`; missing optional equipment context → `200` with `needs_clarification`. The clarification branch runs before retrieval and generation. Trace it in `DEMO/app/service.py`.

In [ ]:
print("Supported request from §1:", diagnosis.status_code, diagnosis.json().get("status"))
for label, payload in [
    ("missing required question", {}),
    ("blank question", {"question": "   "}),
    ("missing optional equipment context", {"question": "Which airflow checks are documented?"}),
]:
    result = client.post("/v1/diagnose", json=payload)
    print(label, "→", result.status_code, result.json())

**Explain:** Why is a clarification a successful HTTP response rather than a schema error? Identify the input declaration and the service branch that establish the difference. The `mode` label alone does not prove a provider call happened.

## 3. Extend the scaffold

### Instructor demo: change the audience instructions

The supplied `/practice/diagnose-brief` route asks for three concise technician bullets. It reuses the same request model, response model, evidence store, and model as `/v1/diagnose`. Its complete source is already in `DEMO/app/campus_routes.py`; registration is in `DEMO/app/campus_demo.py`.

Read `BRIEF_PROMPT` and the `system_prompt=BRIEF_PROMPT` binding before running. **Predict:** what should change, and which fields must remain stable? This is an analogous demonstration; your independent task is to design a dispatch handover.

In [ ]:
brief = client.post("/practice/diagnose-brief", json=question)
for label, result in [("Technician", diagnosis), ("Concise technician", brief)]:
    print(label, "→", result.status_code)
    print(json.dumps(result.json(), indent=2))

**Interpret:** compare audience, documented checks, and what remains to verify. Wording may differ for reasons beyond the prompt; one pair is not a quality benchmark. Confirm both successful outputs use `answer`, `status`, `citations`, and `mode`.

### Your independent practice: dispatch handover

Work in `PROJECT`, not `DEMO`. Adapt the supplied route pattern into `PROJECT/app/handover_routes.py`, choose your dispatch prompt, and register it in `PROJECT/app/main.py`. The Campus lesson defines the constraints and two-request comparison. Your handler passes the prompt into `build_diagnosis_graph`, then passes that graph into `run_diagnosis`.

The cell below is off until your route exists. Enable it to start **your** app and send two different supported requests through both routes. A `404` means your route is not registered; it is not demo completion.

In [ ]:
RUN_MY_DISPATCH = False
if RUN_MY_DISPATCH:
    import httpx
    from module_b.runtime import ServiceProcess
    with ServiceProcess(project_dir=PROJECT, startup_timeout=120) as project_service:
        with httpx.Client(base_url=project_service.base_url, timeout=120) as project_client:
            for request_body in [
                question,
                {"question": "The unit runs hot after filter service. Which checks should be verified?", "equipment_id": "EQ-FC-1002"},
            ]:
                for path in ["/v1/diagnose", "/v1/dispatch-handover"]:
                    result = project_client.post(path, json=request_body)
                    print(path, result.status_code, result.json())
else:
    print("Independent practice: edit your project first, then enable RUN_MY_DISPATCH.")

## 4. Contracts for callers

**Predict:** can an extra field, a blank question, or a string in place of a citations list satisfy the contract? Run the actual Pydantic models on these values. The malformed output below is an intentionally broken local example, not a model response.

In [ ]:
from pydantic import ValidationError
from app.schemas import DiagnosticRequest, DiagnosticResponse
for payload in [question, {"question": "   "}, {**question, "unexpected": True}]:
    try:
        print("Accepted input:", DiagnosticRequest.model_validate(payload).model_dump())
    except ValidationError as error:
        print("Rejected input:", error.errors()[0]["loc"], error.errors()[0]["type"])
if diagnosis.status_code == 200:
    print("Actual output shape:", DiagnosticResponse.model_validate(diagnosis.json()).model_dump())
    malformed = {**diagnosis.json(), "citations": "not-a-list"}
    try:
        DiagnosticResponse.model_validate(malformed)
    except ValidationError as error:
        print("Intentionally broken output rejected:", error.errors()[0]["loc"], error.errors()[0]["type"])
else:
    print("The provider request failed; inspect its actual error before checking a success contract.")

**Explain:** a schema rejects a wrong type; it cannot establish that an answer follows the cited source. Record one structural check and one evidence check you would use for your dispatch response.

## 5. Version endpoints

The supplied pilot keeps `/v1/diagnose` and adds `/v2/diagnose`. Inspect `DiagnosticV2Request`, `V2_PROMPT`, and the model binding in `DEMO/app/campus_routes.py`: v1 accepts up to 2,000 question characters; v2 accepts up to 600.

**Predict:** which version accepts 700 characters? These requests deliberately omit equipment context, isolating contract validation from generation. A valid request therefore returns clarification; no generated answer is needed to prove the length boundary.

In [ ]:
short_request = {"question": "Which filter checks are documented?"}
long_request = {"question": "filter " + "x" * 693}
print("Long question length:", len(long_request["question"]))
for path in ["/v1/diagnose", "/v2/diagnose"]:
    for label, payload in [("short", short_request), ("700 characters", long_request)]:
        result = client.post(path, json=payload)
        print(path, label, "→", result.status_code, result.json().get("status"))

### A deprecation label does not remove a route

The following in-memory change marks v1 in OpenAPI, calls it, and restores the original metadata. It neither edits a source file nor changes a caller’s URL. Expect `deprecated=True` while the existing route still returns clarification.

In [ ]:
from fastapi.routing import APIRoute
# Iterate included routers too when FastAPI represents include_router lazily.
from app.routes import router as original_router
diagnostic_route = next(route for route in original_router.routes
                        if isinstance(route, APIRoute) and route.path == "/v1/diagnose")
old_flag = diagnostic_route.deprecated
try:
    diagnostic_route.deprecated = True
    from fastapi import FastAPI
    migration_demo = FastAPI()
    migration_demo.include_router(original_router)
    migration_demo.state.diagnosis_graph = app.state.diagnosis_graph
    print("Deprecated in OpenAPI:", migration_demo.openapi()["paths"]["/v1/diagnose"]["post"]["deprecated"])
    with TestClient(migration_demo) as migration_client:
        result = migration_client.post("/v1/diagnose", json=short_request)
        print("Existing caller:", result.status_code, result.json()["status"])
finally:
    diagnostic_route.deprecated = old_flag

**Interpret:** two short successful calls would not expose the changed length rule. Keep an input that only the old agreement accepts. Deprecation communicates a transition; removal needs evidence that existing callers can migrate.

## 6. Service scaffold checkpoint

### Your assessment: a supervisor briefing

Use the Campus assessment brief. In `PROJECT`, choose a versioned supervisor path, question limit, prompt, and model-setting name; register the operation while preserving your diagnostic and dispatch routes. Reuse `DiagnosticResponse`. Do not copy the instructor’s technician prompt as a supervisor solution.

Before enabling the cell, write your expected statuses, the boundary each input tests, and the evidence needed to judge the generated answer. The cell sends requests to your project; it does not build the route for you.

In [ ]:
RUN_MY_CHECKPOINT = False
SUPERVISOR_PATH = "/v1/supervisor-briefing"  # Match your chosen route.
if RUN_MY_CHECKPOINT:
    import httpx
    from module_b.runtime import ServiceProcess
    with ServiceProcess(project_dir=PROJECT, startup_timeout=120) as project_service:
        with httpx.Client(base_url=project_service.base_url, timeout=120) as project_client:
            for path, payload in [
                ("/v1/diagnose", question),
                ("/v1/dispatch-handover", question),
                (SUPERVISOR_PATH, question),
                (SUPERVISOR_PATH, {"question": ""}),
            ]:
                result = project_client.post(path, json=payload)
                print(path, result.status_code, result.json())
                if path == SUPERVISOR_PATH and result.status_code == 200:
                    print("Output schema:", DiagnosticResponse.model_validate(result.json()).model_dump())
else:
    print("Assessment has not been attempted. Complete your route and enable RUN_MY_CHECKPOINT.")

**Your handoff:** identify the route and contract; quote only an actual observed answer; compare its claims with cited evidence; explain one remaining limitation. Keep predictions and observations in a text cell in your personal notebook. Running the instructor demo alone does not satisfy these requirements.

## Save your Campus checkpoint

Save the notebook in Drive to retain predictions, outputs, and notes. The export below packages **your `PROJECT` source and data**, not the instructor demo. If you only ran the demos, it contains an unchanged starter. Review your editable files before sharing; an export allowlist cannot detect a credential pasted into Python code.

Download the printed ZIP using Colab’s Files panel. In the next sprint, set `CHECKPOINT` to its uploaded path and use a new `PROJECT` folder. For a same-runtime rerun after restoring, clear `CHECKPOINT` to reuse the existing project. Never overwrite an existing workspace to recover from an error.

In [ ]:
if globals().get("client") is not None:
    client.__exit__(None, None, None)
    client = None
CHECKPOINT_OUT = save_checkpoint(PROJECT, BASE / "exports", sprint=1)
print("Your project checkpoint:", CHECKPOINT_OUT)
print("Save the notebook separately for observations and any safe log records you displayed.")

## Watch out for

- **Missing package after setup:** reopen the original Campus link for the updated setup. It loads a versioned source folder and refreshes an older imported package without overwriting your project. Run setup, then rerun the demonstration sections in order. Preserve your saved notebook observations.
- **Provider error or quota limit:** keep the actual error, check your provider configuration, and rerun only the affected call. Do not substitute a sample answer.
- **Changing Python files has no effect:** stop the owned process and restart it. The supplied demo and your `app.main` are different applications; check which one the request cell starts.
- **Checkpoint destination already exists:** clear `CHECKPOINT` to reuse it, or choose a new `PROJECT` path to restore. Your existing files are preserved.

## Summary

Use the observations above to explain this sprint’s service boundaries. Complete your own project tasks using the lesson brief, then keep its source and run evidence together for the next sprint.